# Auditoria de Integridade do Dataset WikiArt

Este notebook executa a auditoria técnica das imagens candidatas ao experimento de classificação de movimentos artísticos.

A análise é restrita às quatro macroclasses definidas para o TCC:

- Renaissance
- Baroque
- Impressionism
- Cubism

Os objetivos desta etapa são:

1. verificar abertura e decodificação das imagens;
2. validar dimensões e compatibilidade com patches de 256×256 pixels;
3. caracterizar largura, altura, resolução digital e orientação;
4. identificar duplicatas exatas por SHA-256 seguido de comparação pixel a pixel;
5. investigar variantes digitais da mesma obra por hash perceptual baseado em DCT;
6. consolidar grupos de duplicação e registrar descartes de forma reproduzível.

Nenhuma obra é descartada silenciosamente. Toda exclusão deve possuir um motivo explícito e permanecer rastreável pelo `Indice_Dataset`.

> **Observação:** neste notebook, “resolução” refere-se à quantidade de pixels da imagem digital (`largura × altura`), e não a DPI/PPI ou à resolução física da obra.

## 1. Configuração do ambiente

As definições de rótulos e macroclasses permanecem centralizadas em `src/data/labels.py`.

O notebook apenas importa essas definições e cria os caminhos necessários para os artefatos de auditoria.

In [ ]:
from pathlib import Path
import hashlib
import sys
from collections import defaultdict
from itertools import combinations, product

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from datasets import load_dataset


PATCH_SIZE = 256
LIMIAR_PHASH = 4


def encontrar_raiz_projeto(inicio: Path = Path.cwd()) -> Path:
    """Localiza a raiz do projeto procurando pela pasta src/."""
    for caminho in [inicio, *inicio.parents]:
        if (caminho / "src").is_dir():
            return caminho

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto a partir do diretório atual."
    )


RAIZ_PROJETO = encontrar_raiz_projeto()

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))


from src.data.labels import (
    CLASSES_FINAIS,
    ESTILOS_ORIGINAIS_EXPERIMENTO,
    MAPA_MACROCLASSES,
)


PASTA_METADATA = RAIZ_PROJETO / "data" / "metadata"
PASTA_METADATA.mkdir(parents=True, exist_ok=True)

CAMINHO_AUDITORIA = PASTA_METADATA / "auditoria_integridade.csv"
CAMINHO_QUASE_DUPLICATAS = PASTA_METADATA / "candidatos_quase_duplicatas.csv"
CAMINHO_GRUPOS_DUPLICATAS = PASTA_METADATA / "grupos_duplicatas_confirmadas.csv"
CAMINHO_DESCARTES = PASTA_METADATA / "descartes_auditoria.csv"

print(f"Raiz do projeto: {RAIZ_PROJETO}")

## 2. Carregamento das obras candidatas

O WikiArt completo contém 81.444 obras. Entretanto, a auditoria é restrita aos registros pertencentes aos estilos originais que compõem as quatro macroclasses do experimento.

O campo `Indice_Dataset` preserva a posição original de cada obra na partição `train`, permitindo rastrear posteriormente qualquer exclusão até o registro original do WikiArt.

In [ ]:
dataset = load_dataset("huggan/wikiart")
train = dataset["train"]

style_names = train.features["style"].names

df_auditoria = pd.DataFrame({
    "Indice_Dataset": range(len(train)),
    "style_id": train["style"],
})

df_auditoria["Estilo"] = df_auditoria["style_id"].map(
    lambda style_id: style_names[style_id]
)

df_auditoria = df_auditoria[
    df_auditoria["Estilo"].isin(ESTILOS_ORIGINAIS_EXPERIMENTO)
].copy()

df_auditoria["Macro_Estilo"] = df_auditoria["Estilo"].map(
    MAPA_MACROCLASSES
)

df_auditoria = df_auditoria.reset_index(drop=True)

print(f"WikiArt completo: {len(train):,} obras")
print(f"Obras candidatas: {len(df_auditoria):,}")

display(
    df_auditoria["Macro_Estilo"]
    .value_counts()
    .reindex(CLASSES_FINAIS)
    .rename("Quantidade")
    .to_frame()
)

## 3. Verificação de abertura e decodificação das imagens

Uma imagem presente no dataset não é necessariamente utilizável apenas porque existe um registro correspondente nos metadados.

Nesta etapa, cada obra candidata é carregada e sua decodificação completa é forçada. Eventuais exceções são registradas juntamente com o índice original da obra e o tipo do erro encontrado.

Durante a mesma leitura também são coletadas largura, altura, modo de cor e formato do arquivo. Esses atributos serão analisados nas próximas etapas, evitando a necessidade de decodificar todas as imagens novamente apenas para recuperar suas dimensões.

Nenhuma exclusão é realizada nesta etapa; o objetivo é apenas produzir o diagnóstico.

In [ ]:
resultados_auditoria = []
total = len(df_auditoria)

for posicao, linha in df_auditoria.iterrows():
    indice_dataset = int(linha["Indice_Dataset"])

    resultado = {
        "Indice_Dataset": indice_dataset,
        "Estilo": linha["Estilo"],
        "Macro_Estilo": linha["Macro_Estilo"],
        "Status_Decodificacao": "OK",
        "Largura": None,
        "Altura": None,
        "Modo_Cor": None,
        "Formato": None,
        "Tipo_Erro": None,
        "Mensagem_Erro": None,
    }

    try:
        imagem = train[indice_dataset]["image"]

        if imagem is None:
            raise ValueError("Imagem ausente no registro.")

        imagem.load()
        largura, altura = imagem.size

        resultado["Largura"] = largura
        resultado["Altura"] = altura
        resultado["Modo_Cor"] = imagem.mode
        resultado["Formato"] = imagem.format

    except Exception as erro:
        resultado["Status_Decodificacao"] = "ERRO"
        resultado["Tipo_Erro"] = type(erro).__name__
        resultado["Mensagem_Erro"] = str(erro)

    resultados_auditoria.append(resultado)

    if (posicao + 1) % 500 == 0:
        print(f"{posicao + 1:,} / {total:,} imagens verificadas")


df_integridade = pd.DataFrame(resultados_auditoria)

In [ ]:
resumo_decodificacao = (
    df_integridade["Status_Decodificacao"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Quantidade")
)

df_erros_decodificacao = df_integridade[
    df_integridade["Status_Decodificacao"] == "ERRO"
].copy()

resumo_decodificacao_por_classe = pd.crosstab(
    df_integridade["Macro_Estilo"],
    df_integridade["Status_Decodificacao"],
).reindex(CLASSES_FINAIS)

display(resumo_decodificacao)
display(resumo_decodificacao_por_classe)

print(f"Erros de decodificação: {len(df_erros_decodificacao):,}")

if not df_erros_decodificacao.empty:
    display(df_erros_decodificacao)

### Resultado de referência

Na execução realizada durante a auditoria, as 26.426 imagens candidatas foram decodificadas com sucesso. Nenhum registro foi descartado por falha de abertura ou decodificação.

As etapas seguintes assumem imagens decodificáveis. Caso uma execução futura encontre erros, esses casos devem ser tratados antes de prosseguir.

In [ ]:
if not df_erros_decodificacao.empty:
    raise RuntimeError(
        "Foram encontrados erros de decodificação. "
        "Revise esses registros antes de prosseguir com a auditoria."
    )

## 4. Verificação de dimensões inválidas

Após confirmar que todas as imagens candidatas puderam ser decodificadas, são avaliadas suas dimensões.

Uma imagem é considerada dimensionalmente inválida quando:

- a largura ou a altura não foi registrada;
- a largura é menor ou igual a zero;
- a altura é menor ou igual a zero.

Nesta etapa ainda não é aplicado o requisito mínimo de 256×256 pixels. Esse critério será analisado separadamente na próxima etapa da auditoria.

Nenhuma obra é descartada silenciosamente. Os casos identificados são registrados para posterior consolidação dos descartes.

In [ ]:
mascara_dimensao_invalida = (
    df_integridade["Largura"].isna()
    | df_integridade["Altura"].isna()
    | (df_integridade["Largura"] <= 0)
    | (df_integridade["Altura"] <= 0)
)

df_integridade["Dimensao_Valida"] = ~mascara_dimensao_invalida

df_dimensoes_invalidas = df_integridade[
    ~df_integridade["Dimensao_Valida"]
].copy()

print(
    f"Imagens com dimensões válidas: "
    f"{df_integridade['Dimensao_Valida'].sum():,}"
)
print(f"Imagens com dimensões inválidas: {len(df_dimensoes_invalidas):,}")

display(df_integridade[["Largura", "Altura"]].describe())

## 5. Verificação do tamanho mínimo para geração de patches

O pipeline metodológico prevê a geração de patches de 256×256 pixels.

Nesta etapa, verifica-se se alguma obra candidata possui largura ou altura inferior a 256 pixels, condição que impediria a extração direta de pelo menos um patch completo sem redimensionamento ou preenchimento artificial.

Embora a análise descritiva das dimensões já indique valores mínimos superiores a esse limite, o critério é registrado explicitamente para garantir rastreabilidade e reprodutibilidade da auditoria.

In [ ]:
df_integridade["Menor_Que_Patch"] = (
    (df_integridade["Largura"] < PATCH_SIZE)
    | (df_integridade["Altura"] < PATCH_SIZE)
)

df_menores_patch = df_integridade[
    df_integridade["Menor_Que_Patch"]
].copy()

resumo_tamanho_minimo = pd.crosstab(
    df_integridade["Macro_Estilo"],
    df_integridade["Menor_Que_Patch"],
).reindex(CLASSES_FINAIS).rename(
    columns={
        False: "Adequada",
        True: "Menor_Que_256",
    }
)

print(
    f"Imagens menores que {PATCH_SIZE}x{PATCH_SIZE}: "
    f"{len(df_menores_patch):,}"
)
display(resumo_tamanho_minimo)

## 6. Distribuição das dimensões e resolução das imagens

Após verificar a validade dimensional e o tamanho mínimo necessário para a geração de patches, esta etapa analisa a distribuição das dimensões das imagens candidatas.

São avaliados:

- largura em pixels;
- altura em pixels;
- quantidade total de pixels da imagem;
- proporção entre largura e altura.

Neste contexto, o termo resolução refere-se à quantidade de pixels da imagem digital (`largura × altura`) e não à resolução física da obra ou da digitalização em DPI/PPI.

Essa análise permite identificar padrões de padronização do dataset, valores extremos e diferenças sistemáticas entre as macroclasses.

Também é registrada a orientação da imagem como:

- `Horizontal`: largura maior que altura;
- `Vertical`: altura maior que largura;
- `Quadrada`: dimensões iguais.

Essa nomenclatura evita confundir a orientação geométrica da imagem com o campo de gênero artístico `landscape`.

In [ ]:
df_integridade["Total_Pixels"] = (
    df_integridade["Largura"] * df_integridade["Altura"]
)

df_integridade["Megapixels"] = (
    df_integridade["Total_Pixels"] / 1_000_000
).round(2)

df_integridade["Proporcao"] = (
    df_integridade["Largura"] / df_integridade["Altura"]
).round(3)

df_integridade["Orientacao"] = np.select(
    [
        df_integridade["Largura"] > df_integridade["Altura"],
        df_integridade["Largura"] < df_integridade["Altura"],
    ],
    [
        "Horizontal",
        "Vertical",
    ],
    default="Quadrada",
)

resumo_dimensoes = df_integridade[
    ["Largura", "Altura", "Megapixels", "Proporcao"]
].describe().round(2)

display(resumo_dimensoes)

In [ ]:
resumo_por_macroclasse = (
    df_integridade
    .groupby("Macro_Estilo")
    .agg(
        Quantidade=("Indice_Dataset", "size"),
        Largura_Media=("Largura", "mean"),
        Largura_Mediana=("Largura", "median"),
        Altura_Media=("Altura", "mean"),
        Altura_Mediana=("Altura", "median"),
        Megapixels_Media=("Megapixels", "mean"),
        Megapixels_Mediana=("Megapixels", "median"),
        Proporcao_Media=("Proporcao", "mean"),
    )
    .reindex(CLASSES_FINAIS)
    .round(2)
)

orientacao_por_classe = (
    pd.crosstab(
        df_integridade["Macro_Estilo"],
        df_integridade["Orientacao"],
        normalize="index",
    )
    .mul(100)
    .reindex(CLASSES_FINAIS)
    .round(2)
)

display(resumo_por_macroclasse)
display(orientacao_por_classe)

In [ ]:
for coluna, titulo, eixo_x in [
    ("Largura", "Distribuição da largura das imagens", "Largura (pixels)"),
    ("Altura", "Distribuição da altura das imagens", "Altura (pixels)"),
    (
        "Megapixels",
        "Distribuição da resolução digital das imagens",
        "Resolução aproximada (megapixels)",
    ),
]:
    plt.figure(figsize=(10, 5))
    plt.hist(df_integridade[coluna], bins=50)
    plt.title(titulo)
    plt.xlabel(eixo_x)
    plt.ylabel("Quantidade de imagens")
    plt.tight_layout()
    plt.show()

In [ ]:
larguras_frequentes = (
    df_integridade["Largura"]
    .value_counts()
    .head(15)
    .rename_axis("Largura")
    .reset_index(name="Quantidade")
)

alturas_frequentes = (
    df_integridade["Altura"]
    .value_counts()
    .head(15)
    .rename_axis("Altura")
    .reset_index(name="Quantidade")
)

dimensoes_frequentes = (
    df_integridade
    .groupby(["Largura", "Altura"])
    .size()
    .reset_index(name="Quantidade")
    .sort_values("Quantidade", ascending=False)
    .head(20)
)

display(larguras_frequentes)
display(alturas_frequentes)
display(dimensoes_frequentes)

In [ ]:
maiores_imagens = (
    df_integridade
    .nlargest(20, "Total_Pixels")
    [
        [
            "Indice_Dataset",
            "Macro_Estilo",
            "Largura",
            "Altura",
            "Megapixels",
        ]
    ]
)

display(maiores_imagens)

### Observações da execução de referência

A auditoria mostrou forte recorrência de dimensões envolvendo aproximadamente 1.382 pixels, sugerindo padronização ou limitação prévia das reproduções digitais. A origem exata desse processamento não pode ser inferida apenas a partir dos dados.

As quatro macroclasses apresentaram valores médios de resolução próximos. Entretanto, a orientação geométrica mostrou associação relevante com a classe: o Impressionismo apresentou predominância horizontal, enquanto Renascimento, Barroco e Cubismo apresentaram predominância vertical. Essa diferença deve ser considerada posteriormente como possível fonte de correlação espúria.

## 7. Investigação de duplicatas exatas

Nesta etapa são investigadas imagens digitalmente idênticas entre as obras candidatas.

A detecção é realizada em duas fases:

1. cálculo de um hash SHA-256 a partir das dimensões e dos pixels RGB decodificados de cada imagem;
2. confirmação dos grupos com hashes repetidos por comparação exata dos dados de pixels.

O hash é utilizado apenas como mecanismo eficiente de seleção de candidatos. Uma coincidência de hash não é tratada isoladamente como prova de duplicação.

A comparação é realizada sobre os pixels decodificados, e não sobre os bytes originais do arquivo, para evitar que diferenças irrelevantes de metadados ou codificação impeçam a identificação de imagens visualmente idênticas.

Nenhuma imagem é removida nesta etapa. Os casos confirmados são apenas registrados para posterior decisão de descarte.

In [ ]:
def calcular_hash_pixels(imagem):
    """Calcula SHA-256 das dimensões e dos pixels RGB decodificados."""
    imagem_rgb = imagem.convert("RGB")
    largura, altura = imagem_rgb.size

    hash_sha = hashlib.sha256()
    hash_sha.update(f"{largura}x{altura}".encode("utf-8"))
    hash_sha.update(imagem_rgb.tobytes())

    return hash_sha.hexdigest()


hashes = []
total = len(df_integridade)

for posicao, linha in df_integridade.iterrows():
    indice_dataset = int(linha["Indice_Dataset"])
    imagem = train[indice_dataset]["image"]

    hashes.append(calcular_hash_pixels(imagem))

    if (posicao + 1) % 500 == 0:
        print(f"{posicao + 1:,} / {total:,} hashes calculados")

df_integridade["Hash_SHA256_Pixels"] = hashes

In [ ]:
contagem_hashes = df_integridade[
    "Hash_SHA256_Pixels"
].value_counts()

hashes_repetidos = contagem_hashes[
    contagem_hashes > 1
]

df_candidatas_duplicatas = (
    df_integridade[
        df_integridade["Hash_SHA256_Pixels"].isin(hashes_repetidos.index)
    ]
    .sort_values(["Hash_SHA256_Pixels", "Indice_Dataset"])
    .copy()
)

print(
    f"Hashes únicos: "
    f"{df_integridade['Hash_SHA256_Pixels'].nunique():,}"
)
print(f"Hashes repetidos: {len(hashes_repetidos):,}")
print(
    "Imagens pertencentes a grupos candidatos: "
    f"{int(hashes_repetidos.sum()):,}"
)

In [ ]:
df_integridade["Duplicata_Exata"] = False
df_integridade["Grupo_Duplicata_Exata"] = pd.NA

numero_grupo = 1

for _, grupo in df_candidatas_duplicatas.groupby("Hash_SHA256_Pixels"):
    indices = grupo["Indice_Dataset"].astype(int).tolist()

    referencia = indices[0]
    pixels_referencia = np.asarray(
        train[referencia]["image"].convert("RGB")
    )

    confirmados = [referencia]

    for indice in indices[1:]:
        pixels_comparados = np.asarray(
            train[indice]["image"].convert("RGB")
        )

        mesma_imagem = (
            pixels_referencia.shape == pixels_comparados.shape
            and np.array_equal(pixels_referencia, pixels_comparados)
        )

        if mesma_imagem:
            confirmados.append(indice)

    if len(confirmados) > 1:
        grupo_id = f"DUP_EXATA_{numero_grupo:04d}"

        mascara = df_integridade["Indice_Dataset"].isin(confirmados)

        df_integridade.loc[mascara, "Duplicata_Exata"] = True
        df_integridade.loc[
            mascara,
            "Grupo_Duplicata_Exata",
        ] = grupo_id

        numero_grupo += 1


df_duplicatas_exatas = df_integridade[
    df_integridade["Duplicata_Exata"]
].copy()

print(
    "Grupos de duplicatas exatas confirmadas: "
    f"{df_duplicatas_exatas['Grupo_Duplicata_Exata'].nunique():,}"
)
print(
    "Imagens em grupos de duplicatas exatas: "
    f"{len(df_duplicatas_exatas):,}"
)

In [ ]:
if df_duplicatas_exatas.empty:
    df_duplicatas_exatas_multiclasse = pd.DataFrame()
else:
    grupos_exatos_multiclasse = (
        df_duplicatas_exatas
        .groupby("Grupo_Duplicata_Exata")["Macro_Estilo"]
        .nunique()
    )

    grupos_exatos_multiclasse = grupos_exatos_multiclasse[
        grupos_exatos_multiclasse > 1
    ].index

    df_duplicatas_exatas_multiclasse = (
        df_duplicatas_exatas[
            df_duplicatas_exatas["Grupo_Duplicata_Exata"].isin(
                grupos_exatos_multiclasse
            )
        ]
        .sort_values(["Grupo_Duplicata_Exata", "Macro_Estilo"])
    )

print(
    "Grupos exatos envolvendo mais de uma macroclasse: "
    f"{df_duplicatas_exatas_multiclasse['Grupo_Duplicata_Exata'].nunique() if not df_duplicatas_exatas_multiclasse.empty else 0}"
)

display(df_duplicatas_exatas_multiclasse)

## 8. Investigação de quase-duplicatas digitais

A detecção de duplicatas exatas por SHA-256 identificou imagens com pixels completamente idênticos. Entretanto, diferentes arquivos podem representar a mesma obra após operações como redimensionamento, recompressão ou pequenas alterações digitais.

Para investigar esses casos, utiliza-se um hash perceptual baseado em DCT (pHash). Diferentemente do SHA-256, o pHash busca preservar similaridade entre imagens visualmente próximas.

Nesta análise:

1. é calculado um pHash de 64 bits para cada imagem;
2. são procurados pares cuja distância de Hamming seja igual ou inferior a 4;
3. duplicatas exatas previamente identificadas são retiradas dessa lista;
4. os pares restantes são tratados apenas como candidatos;
5. nenhuma exclusão é realizada sem confirmação posterior.

O limiar utilizado constitui uma heurística de triagem e não uma prova de duplicação. Obras distintas podem apresentar hashes perceptualmente próximos, especialmente dentro de um mesmo movimento artístico.

In [ ]:
def calcular_phash64(
    imagem,
    tamanho_hash=8,
    fator_alta_frequencia=4,
):
    """
    Calcula um hash perceptual de 64 bits baseado em DCT.

    Retorna:
        hash_int: representação inteira usada na busca por distância;
        hash_hex: representação hexadecimal usada para registro.
    """
    tamanho = tamanho_hash * fator_alta_frequencia

    imagem_cinza = (
        imagem
        .convert("L")
        .resize(
            (tamanho, tamanho),
            Image.Resampling.LANCZOS,
        )
    )

    matriz = np.asarray(imagem_cinza, dtype=np.float32)
    dct = cv2.dct(matriz)

    baixas_frequencias = dct[
        :tamanho_hash,
        :tamanho_hash,
    ]

    mediana = np.median(
        baixas_frequencias.flatten()[1:]
    )

    bits = baixas_frequencias > mediana

    hash_int = 0

    for bit in bits.flatten():
        hash_int = (hash_int << 1) | int(bit)

    hash_hex = f"{hash_int:016x}"

    return hash_int, hash_hex

In [ ]:
phashes_int = []
phashes_hex = []

total = len(df_integridade)

for posicao, linha in df_integridade.iterrows():
    indice_dataset = int(linha["Indice_Dataset"])
    imagem = train[indice_dataset]["image"]

    hash_int, hash_hex = calcular_phash64(imagem)

    phashes_int.append(hash_int)
    phashes_hex.append(hash_hex)

    if (posicao + 1) % 500 == 0:
        print(f"{posicao + 1:,} / {total:,} pHashes calculados")


df_integridade["_PHash64_Int"] = phashes_int
df_integridade["PHash64"] = phashes_hex

In [ ]:
def distancia_hamming(hash_a, hash_b):
    """Distância de Hamming entre dois inteiros de 64 bits."""
    return (hash_a ^ hash_b).bit_count()


class BKTree:
    """BK-tree simples para busca de hashes por distância de Hamming."""

    def __init__(self, funcao_distancia):
        self.funcao_distancia = funcao_distancia
        self.raiz = None

    def adicionar(self, valor):
        if self.raiz is None:
            self.raiz = {
                "valor": valor,
                "filhos": {},
            }
            return

        no = self.raiz

        while True:
            distancia = self.funcao_distancia(
                valor,
                no["valor"],
            )

            if distancia == 0:
                return

            if distancia in no["filhos"]:
                no = no["filhos"][distancia]
            else:
                no["filhos"][distancia] = {
                    "valor": valor,
                    "filhos": {},
                }
                return

    def buscar(self, valor, distancia_maxima):
        if self.raiz is None:
            return []

        resultados = []
        pilha = [self.raiz]

        while pilha:
            no = pilha.pop()

            distancia = self.funcao_distancia(
                valor,
                no["valor"],
            )

            if distancia <= distancia_maxima:
                resultados.append(
                    (distancia, no["valor"])
                )

            limite_inferior = distancia - distancia_maxima
            limite_superior = distancia + distancia_maxima

            for distancia_aresta, filho in no["filhos"].items():
                if (
                    limite_inferior
                    <= distancia_aresta
                    <= limite_superior
                ):
                    pilha.append(filho)

        return resultados

In [ ]:
indices_por_phash = defaultdict(list)

for indice_df, linha in df_integridade.iterrows():
    indices_por_phash[
        linha["_PHash64_Int"]
    ].append(indice_df)

phashes_unicos = list(indices_por_phash.keys())

arvore_phash = BKTree(distancia_hamming)

for phash in phashes_unicos:
    arvore_phash.adicionar(phash)

print(f"pHashes únicos: {len(phashes_unicos):,}")
print("BK-tree construída.")

In [ ]:
pares_candidatos = []
pares_processados = set()

for phash_a in phashes_unicos:
    vizinhos = arvore_phash.buscar(
        phash_a,
        LIMIAR_PHASH,
    )

    for distancia, phash_b in vizinhos:
        chave_hash = tuple(sorted((phash_a, phash_b)))

        if chave_hash in pares_processados:
            continue

        pares_processados.add(chave_hash)

        indices_a = indices_por_phash[phash_a]
        indices_b = indices_por_phash[phash_b]

        if phash_a == phash_b:
            pares_indices = combinations(indices_a, 2)
        else:
            pares_indices = product(indices_a, indices_b)

        for indice_df_a, indice_df_b in pares_indices:
            linha_a = df_integridade.loc[indice_df_a]
            linha_b = df_integridade.loc[indice_df_b]

            if (
                linha_a["Hash_SHA256_Pixels"]
                == linha_b["Hash_SHA256_Pixels"]
            ):
                continue

            pares_candidatos.append({
                "Indice_A": int(linha_a["Indice_Dataset"]),
                "Indice_B": int(linha_b["Indice_Dataset"]),
                "Macroclasse_A": linha_a["Macro_Estilo"],
                "Macroclasse_B": linha_b["Macro_Estilo"],
                "Estilo_A": linha_a["Estilo"],
                "Estilo_B": linha_b["Estilo"],
                "Distancia_pHash": distancia,
                "PHash_A": linha_a["PHash64"],
                "PHash_B": linha_b["PHash64"],
                "Mesma_Macroclasse": (
                    linha_a["Macro_Estilo"]
                    == linha_b["Macro_Estilo"]
                ),
            })


df_quase_duplicatas = pd.DataFrame(pares_candidatos)

if not df_quase_duplicatas.empty:
    df_quase_duplicatas = (
        df_quase_duplicatas
        .sort_values(
            ["Distancia_pHash", "Indice_A", "Indice_B"]
        )
        .reset_index(drop=True)
    )

print(
    "Pares candidatos não exatos: "
    f"{len(df_quase_duplicatas):,}"
)

In [ ]:
if df_quase_duplicatas.empty:
    resumo_phash = pd.DataFrame(
        columns=["Distancia_pHash", "Quantidade_Pares"]
    )
    candidatos_multiclasse = pd.DataFrame()
else:
    resumo_phash = (
        df_quase_duplicatas["Distancia_pHash"]
        .value_counts()
        .sort_index()
        .rename_axis("Distancia_pHash")
        .reset_index(name="Quantidade_Pares")
    )

    candidatos_multiclasse = df_quase_duplicatas[
        ~df_quase_duplicatas["Mesma_Macroclasse"]
    ].copy()

display(resumo_phash)

print(
    "Pares candidatos entre macroclasses diferentes: "
    f"{len(candidatos_multiclasse):,}"
)

### Inspeção visual manual

Na execução de referência foram encontrados 136 pares não exatos:

- distância 0: 20 pares;
- distância 2: 49 pares;
- distância 4: 67 pares.

Todos foram inspecionados visualmente. Os 20 pares de distância 0 e os 49 de distância 2 correspondiam à mesma obra. Entre os 67 pares de distância 4, 66 correspondiam à mesma obra e um par foi rejeitado como falso positivo.

Assim, 135 dos 136 candidatos foram confirmados como variantes digitais da mesma obra.

A célula abaixo registra o resultado dessa revisão humana. Ela **não transforma pHash ≤ 4 em regra universal**. Caso uma execução futura produza uma distribuição diferente de candidatos, a inspeção manual deve ser repetida.

In [ ]:
def mostrar_candidatos_phash(
    df,
    inicio=0,
    quantidade=10,
):
    """
    Exibe pares candidatos para inspeção humana.

    A função não é chamada automaticamente para evitar incorporar
    dezenas de figuras ao notebook salvo no repositório.
    """
    fim = min(inicio + quantidade, len(df))

    for i in range(inicio, fim):
        candidato = df.iloc[i]

        indice_a = int(candidato["Indice_A"])
        indice_b = int(candidato["Indice_B"])
        distancia = candidato["Distancia_pHash"]

        imagem_a = train[indice_a]["image"]
        imagem_b = train[indice_b]["image"]

        fig, axes = plt.subplots(
            1,
            2,
            figsize=(10, 5),
        )

        axes[0].imshow(imagem_a)
        axes[0].set_title(
            f"A — índice {indice_a}\n"
            f"{candidato['Macroclasse_A']}"
        )
        axes[0].axis("off")

        axes[1].imshow(imagem_b)
        axes[1].set_title(
            f"B — índice {indice_b}\n"
            f"{candidato['Macroclasse_B']}"
        )
        axes[1].axis("off")

        fig.suptitle(
            f"Candidato {i} — distância pHash = {distancia}"
        )

        plt.tight_layout()
        plt.show()


# Exemplo de reinspeção:
mostrar_candidatos_phash(df_quase_duplicatas, inicio=0, quantidade=10)

In [ ]:
CONTAGENS_REVISAO_REFERENCIA = {
    0: 20,
    2: 49,
    4: 67,
}

PAR_FALSO_POSITIVO = frozenset({
    40177,
    73568,
})

contagens_atuais = (
    df_quase_duplicatas["Distancia_pHash"]
    .value_counts()
    .sort_index()
    .to_dict()
)

if contagens_atuais != CONTAGENS_REVISAO_REFERENCIA:
    raise RuntimeError(
        "O conjunto de candidatos difere da execução manualmente revisada. "
        "Repita a inspeção visual antes de registrar classificações."
    )


df_quase_duplicatas["Classificacao_Manual"] = "MESMA_OBRA"
df_quase_duplicatas["Observacao"] = (
    "Confirmada por inspeção visual manual"
)

mascara_falso_positivo = df_quase_duplicatas.apply(
    lambda linha: frozenset({
        int(linha["Indice_A"]),
        int(linha["Indice_B"]),
    }) == PAR_FALSO_POSITIVO,
    axis=1,
)

if int(mascara_falso_positivo.sum()) != 1:
    raise RuntimeError(
        "O par registrado como falso positivo não foi encontrado exatamente uma vez. "
        "Repita a revisão manual antes de prosseguir."
    )


df_quase_duplicatas.loc[
    mascara_falso_positivo,
    "Classificacao_Manual",
] = "OBRA_DIFERENTE"

df_quase_duplicatas.loc[
    mascara_falso_positivo,
    "Observacao",
] = "Falso positivo do pHash; rejeitado por inspeção visual"

display(
    df_quase_duplicatas["Classificacao_Manual"]
    .value_counts(dropna=False)
)

## 9. Consolidação dos grupos de duplicação

As duplicatas exatas e as variantes digitais confirmadas são unificadas em um grafo:

- cada registro é um nó;
- cada relação confirmada de identidade entre duas imagens é uma aresta;
- cada componente conexo representa um grupo de registros da mesma obra.

Essa consolidação evita contar pares como se fossem descartes independentes. Por exemplo, três registros da mesma obra podem gerar múltiplas comparações, mas representam apenas um grupo com dois registros excedentes.

In [ ]:
grafo_duplicatas = defaultdict(set)


def adicionar_ligacao(a, b):
    a = int(a)
    b = int(b)

    grafo_duplicatas[a].add(b)
    grafo_duplicatas[b].add(a)


for _, grupo in df_duplicatas_exatas.groupby(
    "Grupo_Duplicata_Exata"
):
    indices = (
        grupo["Indice_Dataset"]
        .astype(int)
        .tolist()
    )

    referencia = indices[0]

    for indice in indices[1:]:
        adicionar_ligacao(referencia, indice)


pares_confirmados = df_quase_duplicatas[
    df_quase_duplicatas["Classificacao_Manual"]
    == "MESMA_OBRA"
]

for _, linha in pares_confirmados.iterrows():
    adicionar_ligacao(
        linha["Indice_A"],
        linha["Indice_B"],
    )

In [ ]:
visitados = set()
grupos_unificados = []

for inicio in grafo_duplicatas:
    if inicio in visitados:
        continue

    pilha = [inicio]
    componente = []

    while pilha:
        atual = pilha.pop()

        if atual in visitados:
            continue

        visitados.add(atual)
        componente.append(atual)

        for vizinho in grafo_duplicatas[atual]:
            if vizinho not in visitados:
                pilha.append(vizinho)

    if len(componente) > 1:
        grupos_unificados.append(
            sorted(componente)
        )


print(
    "Grupos únicos de obras duplicadas: "
    f"{len(grupos_unificados):,}"
)
print(
    "Registros envolvidos em duplicações: "
    f"{sum(len(grupo) for grupo in grupos_unificados):,}"
)
print(
    "Registros excedentes: "
    f"{sum(len(grupo) - 1 for grupo in grupos_unificados):,}"
)

In [ ]:
registros_grupos = []

for numero, indices in enumerate(
    grupos_unificados,
    start=1,
):
    grupo_id = f"DUP_FINAL_{numero:04d}"

    for indice in indices:
        linha = df_integridade[
            df_integridade["Indice_Dataset"] == indice
        ].iloc[0]

        registros_grupos.append({
            "Grupo_Duplicata_Final": grupo_id,
            "Indice_Dataset": indice,
            "Estilo": linha["Estilo"],
            "Macro_Estilo": linha["Macro_Estilo"],
            "Largura": linha["Largura"],
            "Altura": linha["Altura"],
            "Total_Pixels": linha["Total_Pixels"],
        })


df_grupos_duplicatas = pd.DataFrame(registros_grupos)

resumo_classes_grupo = (
    df_grupos_duplicatas
    .groupby("Grupo_Duplicata_Final")
    .agg(
        Quantidade_Registros=("Indice_Dataset", "size"),
        Quantidade_Macroclasses=("Macro_Estilo", "nunique"),
        Macroclasses=(
            "Macro_Estilo",
            lambda valores: ", ".join(
                sorted(set(valores))
            ),
        ),
    )
    .reset_index()
)

grupos_multiclasse = resumo_classes_grupo[
    resumo_classes_grupo["Quantidade_Macroclasses"] > 1
]

print(
    "Grupos envolvendo mais de uma macroclasse: "
    f"{len(grupos_multiclasse)}"
)

display(grupos_multiclasse)

## 10. Política de preservação e registro dos descartes

Para cada grupo confirmado, é mantida exatamente uma representação da obra. A escolha segue uma regra objetiva:

1. maior quantidade total de pixels;
2. em empate, maior largura;
3. persistindo o empate, menor `Indice_Dataset`.

Essa regra evita decisões subjetivas e preserva, quando possível, a versão digital com maior quantidade de informação.

Nenhum grupo envolvendo macroclasses diferentes deve ser descartado automaticamente. Na execução de referência, não foram encontrados grupos multiclasse.

In [ ]:
if not grupos_multiclasse.empty:
    raise RuntimeError(
        "Existem grupos de duplicação envolvendo mais de uma macroclasse. "
        "Revise-os manualmente antes de aplicar a política de preservação."
    )


df_grupos_duplicatas["Manter"] = False

for _, grupo in df_grupos_duplicatas.groupby(
    "Grupo_Duplicata_Final"
):
    grupo_ordenado = grupo.sort_values(
        by=[
            "Total_Pixels",
            "Largura",
            "Indice_Dataset",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )

    indice_linha_manter = grupo_ordenado.index[0]

    df_grupos_duplicatas.loc[
        indice_linha_manter,
        "Manter",
    ] = True


print(
    "Imagens preservadas nos grupos: "
    f"{df_grupos_duplicatas['Manter'].sum():,}"
)
print(
    "Registros excedentes: "
    f"{(~df_grupos_duplicatas['Manter']).sum():,}"
)

In [ ]:
indice_mantido_por_grupo = (
    df_grupos_duplicatas[
        df_grupos_duplicatas["Manter"]
    ]
    .set_index("Grupo_Duplicata_Final")[
        "Indice_Dataset"
    ]
    .to_dict()
)

df_grupos_duplicatas["Indice_Obra_Mantida"] = (
    df_grupos_duplicatas[
        "Grupo_Duplicata_Final"
    ].map(indice_mantido_por_grupo)
)

df_descartes = df_grupos_duplicatas[
    ~df_grupos_duplicatas["Manter"]
].copy()

df_descartes["Motivo_Descarte"] = (
    "Duplicata ou variante digital confirmada da mesma obra"
)

df_descartes["Criterio_Preservacao"] = (
    "Maior resolução; empate por largura; "
    "novo empate pelo menor índice do dataset"
)

df_descartes = df_descartes[
    [
        "Indice_Dataset",
        "Estilo",
        "Macro_Estilo",
        "Grupo_Duplicata_Final",
        "Indice_Obra_Mantida",
        "Largura",
        "Altura",
        "Total_Pixels",
        "Motivo_Descarte",
        "Criterio_Preservacao",
    ]
]

In [ ]:
mapa_grupo_final = (
    df_grupos_duplicatas
    .set_index("Indice_Dataset")[
        "Grupo_Duplicata_Final"
    ]
    .to_dict()
)

mapa_indice_mantido = (
    df_grupos_duplicatas
    .set_index("Indice_Dataset")[
        "Indice_Obra_Mantida"
    ]
    .to_dict()
)

indices_descartados = set(
    df_descartes["Indice_Dataset"]
)

df_integridade["Grupo_Duplicata_Final"] = (
    df_integridade["Indice_Dataset"]
    .map(mapa_grupo_final)
)

df_integridade["Indice_Obra_Mantida"] = (
    df_integridade["Indice_Dataset"]
    .map(mapa_indice_mantido)
)

df_integridade["Descartada_Auditoria"] = (
    df_integridade["Indice_Dataset"]
    .isin(indices_descartados)
)

df_integridade["Aprovada_Auditoria"] = (
    (df_integridade["Status_Decodificacao"] == "OK")
    & df_integridade["Dimensao_Valida"]
    & ~df_integridade["Menor_Que_Patch"]
    & ~df_integridade["Descartada_Auditoria"]
)

df_integridade["Motivo_Descarte"] = pd.NA

df_integridade.loc[
    df_integridade["Descartada_Auditoria"],
    "Motivo_Descarte",
] = (
    "Duplicata ou variante digital confirmada da mesma obra"
)

## 11. Persistência dos artefatos

São produzidos quatro arquivos de rastreabilidade:

- `auditoria_integridade.csv`: diagnóstico completo de todas as obras candidatas;
- `candidatos_quase_duplicatas.csv`: pares perceptuais encontrados e respectiva classificação manual;
- `grupos_duplicatas_confirmadas.csv`: composição dos grupos consolidados e registro da versão preservada;
- `descartes_auditoria.csv`: somente os registros efetivamente excluídos e seus motivos.

A coluna inteira `_PHash64_Int` é usada apenas internamente pela BK-tree. O valor hexadecimal `PHash64` é suficiente para o registro persistente.

In [ ]:
df_integridade_exportacao = df_integridade.drop(
    columns=["_PHash64_Int"]
)

df_integridade_exportacao.to_csv(
    CAMINHO_AUDITORIA,
    index=False,
    encoding="utf-8",
)

df_quase_duplicatas.to_csv(
    CAMINHO_QUASE_DUPLICATAS,
    index=False,
    encoding="utf-8",
)

df_grupos_duplicatas.to_csv(
    CAMINHO_GRUPOS_DUPLICATAS,
    index=False,
    encoding="utf-8",
)

df_descartes.to_csv(
    CAMINHO_DESCARTES,
    index=False,
    encoding="utf-8",
)

print(f"Auditoria completa: {CAMINHO_AUDITORIA}")
print(f"Candidatos pHash: {CAMINHO_QUASE_DUPLICATAS}")
print(f"Grupos confirmados: {CAMINHO_GRUPOS_DUPLICATAS}")
print(f"Descartes: {CAMINHO_DESCARTES}")

## 12. Resumo final da auditoria

O resumo abaixo é calculado diretamente a partir dos objetos produzidos no notebook. Ele deve ser usado como conferência antes de iniciar a etapa de curadoria e balanceamento do subconjunto final.

In [ ]:
resumo_final = pd.DataFrame({
    "Metrica": [
        "Obras candidatas",
        "Erros de decodificação",
        "Dimensões inválidas",
        "Imagens menores que 256x256",
        "Grupos de duplicatas exatas",
        "Imagens em duplicatas exatas",
        "Pares candidatos pHash não exatos",
        "Pares pHash confirmados como mesma obra",
        "Falsos positivos pHash",
        "Grupos finais de obras duplicadas",
        "Registros envolvidos em duplicações",
        "Registros descartados",
        "Grupos finais multiclasse",
        "Obras aprovadas após auditoria",
    ],
    "Valor": [
        len(df_integridade),
        len(df_erros_decodificacao),
        len(df_dimensoes_invalidas),
        len(df_menores_patch),
        df_duplicatas_exatas["Grupo_Duplicata_Exata"].nunique(),
        len(df_duplicatas_exatas),
        len(df_quase_duplicatas),
        (
            df_quase_duplicatas["Classificacao_Manual"]
            .eq("MESMA_OBRA")
            .sum()
        ),
        (
            df_quase_duplicatas["Classificacao_Manual"]
            .eq("OBRA_DIFERENTE")
            .sum()
        ),
        len(grupos_unificados),
        sum(len(grupo) for grupo in grupos_unificados),
        len(df_descartes),
        len(grupos_multiclasse),
        int(df_integridade["Aprovada_Auditoria"].sum()),
    ],
})

display(resumo_final)